### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
# Replace Your_Dir with your path, e.g. /content/drive/MyDrive/Colab Notebooks/EMG_keyboard_NN
%cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: TDS training (for report)

- Ensure the dataset is in `Your_Dir/emg2qwerty/data`.
- Run **one** of the three cells below (Run 1, 2, or 3). Logs and checkpoints go to `logs/YYYY-MM-DD/HH-MM-SS/...`.
- **Run 1 — TDS baseline:** TDS + CTC, no noise/gain augmentation (`transforms=log_spectrogram_baseline`).
- **Run 2 — TDS + new augmentations:** Same model with noise and gain scaling (default `log_spectrogram`).
- **Run 3 — TDS + CR-CTC:** TDS with CR-CTC loss (default transforms with noise and gain).

#### Run 1: TDS baseline (CTC, no noise/gain)

Checkpoints: `logs/.../checkpoints/`.

In [ ]:
# Run 1: TDS baseline (plain CTC, baseline transforms: no noise, no gain)
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline

#### Run 2: TDS + new augmentations (noise, gain)

Same TDS+CTC with default transforms (noise + gain scaling).

In [ ]:
# Run 2: TDS + new augmentations (noise, gain scaling)
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc

#### Run 3: TDS + CR-CTC

TDS with CR-CTC loss; default transforms (noise + gain).

In [ ]:
# Run 3: TDS + CR-CTC
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_crctc

### Step 5: Ablation studies (TDS baseline + CTC)

Use **model=tds_conv_ctc** and **transforms=log_spectrogram_baseline** for both ablations. Run one cell per experiment; compare val/CER across runs for your report.

- **Channel ablation:** How many electrode channels are needed? Run with **electrode_channels=4, 8, 12, 16** (override at CLI).
- **Data ablation:** How much training data is needed? Run with **user=single_user_2sessions**, **single_user_5sessions**, **single_user_7sessions**, or **single_user** (full).

#### Channel ablation: number of electrode channels vs CER

Run one of the four cells below (4, 8, 12, or 16 channels). Default is 16.

In [5]:
# Channel ablation: 4 channels
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline electrode_channels=4

^C


In [ ]:
# Channel ablation: 8 channels
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline electrode_channels=8

In [ ]:
# Channel ablation: 12 channels
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline electrode_channels=12

In [ ]:
# Channel ablation: 16 channels (full)
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline electrode_channels=16

#### Data ablation: amount of training data vs CER

Run one of the four cells below (2, 5, 7, or 14 train sessions). Val/test sessions are unchanged.

In [ ]:
# Data ablation: 2 train sessions
!python -m emg2qwerty.train user=single_user_2sessions cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline

In [ ]:
# Data ablation: 5 train sessions
!python -m emg2qwerty.train user=single_user_5sessions cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline

In [ ]:
# Data ablation: 7 train sessions
!python -m emg2qwerty.train user=single_user_7sessions cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline

In [ ]:
# Data ablation: full data (14 train sessions, same as single_user)
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline

### Step 6: CNN Transformer (250 epochs)

Transformer benefits from longer training. Run **one** of the two cells below (CTC or CR-CTC); both use **trainer.max_epochs=250**.

#### Run 1: CNN Transformer + CTC (plain CTC)

250 epochs; default transforms.

In [ ]:
# CNN Transformer + CTC (plain CTC), 250 epochs
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=cnn_transformer_ctc module.use_cr_ctc=false trainer.max_epochs=250

#### Run 2: CNN Transformer + CR-CTC

250 epochs; default transforms (noise + gain); CR-CTC loss.

In [ ]:
# CNN Transformer + CR-CTC, 250 epochs
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=cnn_transformer_ctc trainer.max_epochs=250

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun